# Chapter 00: Neural Networks, Autograd & Loss Landscapes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vvknyn/self-driving-ai-course/blob/main/notebooks/00_neural_networks_and_autograd.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-Repository-181717.svg)](https://github.com/vvknyn/self-driving-ai-course)

> **The Big Question**: *How does a computational graph tune 1,000,000 synaptic weights simultaneously in milliseconds to learn non-linear stopping distances?*

---

## 1. 🚨 The Real-World Dilemma
In autonomous emergency braking (AEB), kinetic energy is quadratic ($d \propto v^2$). Stacking linear layers without activations mathematically collapses into a single straight line, causing high-speed crashes or parking-lot phantom braking!

In [ ]:
import math
import random
import matplotlib.pyplot as plt
%matplotlib inline

print("Building Karpathy-style Micrograd autograd engine...")

## 2. 🛠️ The Karpathy Build: Micrograd Scalar Autograd Engine
Every scalar value tracks its local gradient, parents in the Directed Acyclic Graph (DAG), and backward chain-rule closure.

In [ ]:
class Value:
    def __init__(self, data, _children=(), _op=""):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")
        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __sub__(self, other):
        return self + (-other)

    def __neg__(self):
        return self * -1.0

    def relu(self):
        out = Value(max(0.0, self.data), (self,), "ReLU")
        def _backward():
            self.grad += (1.0 if self.data > 0 else 0.0) * out.grad
        out._backward = _backward
        return out

    def backward(self):
        topo, visited = [], set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

## 3. 📐 Finite-Difference Numerical Gradient Check (Andrew Ng Style)
Comparing analytical backpropagation gradients to finite differences:
$$\frac{\partial f}{\partial x} \approx \frac{f(x + \epsilon) - f(x - \epsilon)}{2\epsilon}$$

In [ ]:
eps = 1e-5
x = Value(3.0)
w = Value(-2.0)
b = Value(4.0)
y = (x * w + b).relu()
y.backward()
analytical_dw = w.grad

y1 = max(0.0, 3.0 * (-2.0 + eps) + 4.0)
y2 = max(0.0, 3.0 * (-2.0 - eps) + 4.0)
numerical_dw = (y1 - y2) / (2 * eps)
rel_error = abs(analytical_dw - numerical_dw) / (abs(analytical_dw) + abs(numerical_dw) + 1e-8)

print(f"Analytical: {analytical_dw:.6f} | Numerical: {numerical_dw:.6f} | Rel Error: {rel_error:.2e}")
assert rel_error < 1e-5
print("✅ Calculus gradient check certified correct!")

## 4. 🧠 MLP Training on Quadratic Braking Boundary
Training a 2-layer MLP on non-linear stopping distances: $d_{\text{safe}} = 0.05 v^2 + 0.2 v$.

In [ ]:
class Neuron:
    def __init__(self, nin):
        scale = math.sqrt(2.0 / nin)
        self.w = [Value(random.gauss(0, scale)) for _ in range(nin)]
        self.b = Value(0.0)
    def __call__(self, x):
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        return act.relu()
    def parameters(self):
        return self.w + [self.b]

class Layer:
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]
    def __call__(self, x):
        return [n(x) for n in self.neurons]
    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

class MLP:
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]
    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x[0] if len(x) == 1 else x
    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

random.seed(42)
model = MLP(2, [8, 1])

# Generate training data
X_train, y_train = [], []
for _ in range(100):
    v = random.uniform(0.0, 40.0)
    d = random.uniform(0.0, 100.0)
    d_safe = 0.05 * (v**2) + 0.2 * v
    X_train.append([Value(v / 40.0), Value(d / 100.0)])
    y_train.append(Value(1.0 if d < d_safe else 0.0))

losses = []
for step in range(50):
    ypred = [model(x) for x in X_train]
    loss = sum((yp - yt)*(yp - yt) for yp, yt in zip(ypred, y_train)) * (1.0 / len(y_train))
    losses.append(loss.data)
    for p in model.parameters(): p.grad = 0.0
    loss.backward()
    for p in model.parameters(): p.data -= 0.05 * p.grad

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(losses, color="#58a6ff", lw=2)
plt.title("MSE Loss Convergence")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
v_raw = [x[0].data * 40.0 for x in X_train]
d_raw = [x[1].data * 100.0 for x in X_train]
colors = ["#f85149" if y.data > 0.5 else "#3fb950" for y in y_train]
plt.scatter(v_raw, d_raw, c=colors, alpha=0.7, edgecolors="k")
plt.plot(range(41), [0.05*(v**2) + 0.2*v for v in range(41)], color="yellow", lw=2, label="Physics Boundary")
plt.title("Learned Quadratic Braking Space")
plt.xlabel("Speed (m/s)")
plt.ylabel("Distance (m)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. 🩺 Andrew Ng's Diagnostic Field Guide

| Observed Symptom | Underlying Mathematical Mechanism | Verification Test | Production Fix |
| :--- | :--- | :--- | :--- |
| **All weights update identically** | Zero-weight initialization ($W=0$). Symmetry trap prevents differentiation. | Check if `w1.grad == w2.grad`. | Use He/Xavier normal random init. |
| **Loss explodes to NaN** | Learning rate $\eta$ overshoots convex valleys. | Check if $\|\mathbf{g}\| > 100$. | Clip gradient norms ($\|\mathbf{g}\| \le 1.0$) and reduce $\eta$. |
| **Loss plateaus immediately** | Dead ReLU problem (large negative bias). | Count neurons with zero activations. | Use Leaky ReLU or Batch Normalization. |